In [1]:
print("Hello World!")

Hello World!


In [ ]:
def run_lopo_task_with_fi(
        X,
        y,
        groups,
        task_name):

    logo = LeaveOneGroupOut()

    results = []

    all_confusion_matrices = {}

    feature_importance_storage = {
        "SVM": [],
        "Logistic Regression": [],
        "Random Forest": [],
        "KNN": [],
        "XGBoost": []
    }

    trained_models={}

    for model_name, model in get_models().items():

        print(f"\nRunning {task_name} | {model_name}")

        fold_accuracies = []
        fold_precisions = []
        fold_recalls = []
        fold_f1s = []

        overall_y_true = []
        overall_y_pred = []

        fold_counter = 1

        for train_idx, test_idx in logo.split(
                X,
                y,
                groups):

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            scaler = StandardScaler()

            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            model.fit(X_train, y_train)

            # ==========================
            # Feature Importance
            # ==========================

            imp_df = generate_fold_fi_for_lopo(
                model_name ,
                model,
                X_test,
                y_test,
                FEATURES,
                RESULTS / "temp_fi")

            feature_importance_storage[model_name].append(imp_df[["Feature", "Importance"]])

            trained_models[model_name] = model

            # ==========================

            preds = model.predict(X_test)

            fold_accuracies.append(
                accuracy_score(y_test, preds)
            )

            fold_precisions.append(
                precision_score(
                    y_test,
                    preds,
                    average="binary"
                )
            )

            fold_recalls.append(
                recall_score(
                    y_test,
                    preds,
                    average="binary"
                )
            )

            fold_f1s.append(
                f1_score(
                    y_test,
                    preds,
                    average="binary"
                )
            )

            overall_y_true.extend(y_test)
            overall_y_pred.extend(preds)

            fold_counter += 1

        cm = confusion_matrix(
            overall_y_true,
            overall_y_pred
        )

        all_confusion_matrices[model_name] = cm

        results.append({
            "Task": task_name,
            "Model": model_name,
            "Accuracy": np.mean(fold_accuracies),
            "Precision": np.mean(fold_precisions),
            "Recall": np.mean(fold_recalls),
            "F1": np.mean(fold_f1s),
            "Accuracy_STD": np.std(fold_accuracies)
        })

        # ======================================
        # Average feature importance across folds
        # ======================================

        if model_name in feature_importance_storage:

            fold_dfs = feature_importance_storage[model_name]

            merged = pd.concat(fold_dfs)

            avg_importance = (
                merged
                .groupby("Feature")["Importance"]
                .agg(["mean", "std"])
                .reset_index()
                .sort_values("mean", ascending=False)
            )

            avg_importance.to_csv(
                RESULTS /
                f"{task_name}_{model_name}_LOPO_feature_importance.csv",
                index=False
            )

            plt.figure(figsize=(8, 5))

            sns.barplot(
                data=avg_importance,
                x="mean",
                y="Feature"
            )

            plt.title(
                f"{task_name} | {model_name}\nAverage Feature Importance Across LOPO Folds"
            )

            plt.xlabel("Mean Feature Importance")

            plt.tight_layout()

            plt.savefig(
                RESULTS / "task2_feature_importance" /
                f"{task_name}_{model_name}_LOPO_feature_importance.png",
                dpi=300
            )

            plt.close()

            import shutil

            folder_path = RESULTS / "temp_fi"

            try:
                shutil.rmtree(folder_path)
                print("Folder deleted successfully.")
            except FileNotFoundError:
                print("The folder does not exist.")
            except PermissionError:
                print("Permission denied.")

    return (
        pd.DataFrame(results),
        all_confusion_matrices
    )